In [48]:
import os
import json
from datetime import datetime
from pathlib import Path

import wandb
from dotenv import load_dotenv
import numpy as np
import polars as pl
from kaggle.api.kaggle_api_extended import KaggleApi
from kagglesdk.competitions.types.competition_api_service import SubmissionGroup

# Config

In [49]:
load_dotenv(dotenv_path="../../.env")
wandb.login(key=os.environ.get("WANDB_API_KEY"))

RUN_ID = "mlp-057-trl0-5fold-s42"
MESSAGE = "Good Luck!"

COMP = os.environ.get("COMPETITION_NAME")

RUNS = Path("../../runs") / RUN_ID
OUTPUT = Path("../../output")
INPUT = Path("../../input")

MANIFEST_PATH = RUNS / "manifest.json"
SUB_PATH = OUTPUT / f"{RUN_ID}.csv"

api = KaggleApi()
api.authenticate()

with open(MANIFEST_PATH) as f:
    m = json.load(f)

WANDB_ID = m["wandb_id"]

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/hanse/.netrc


# Submit

In [50]:
test = np.load(RUNS / "test.npy")[:250000]

sample_sub = pl.read_csv(INPUT / "sample_submission.csv")

if len(sample_sub) != len(test):
    raise ValueError(
        f"Row mismatch: sample={len(sample_sub)}, test={len(test)}"
    )

sub = sample_sub.with_columns(
    pl.Series("y", test)
)
sub.write_csv(SUB_PATH)

result = api.competition_submit(
    file_name=SUB_PATH,
    message=MESSAGE,
    competition=COMP
)

ref = result.ref

submitted_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Ref: {ref}")

100%|██████████| 4.78M/4.78M [00:01<00:00, 2.97MB/s]


Ref: 47198092


# Public Score

In [51]:
# === Check Public Score ===
subs = api.competition_submissions(
    COMP,
    group=SubmissionGroup.SUBMISSION_GROUP_SUCCESSFUL  # ← 実際に print で確認した正しい名前を指定
)

target = next((s for s in subs if s.ref == ref), None)

if target:
    public = target.public_score
    date = target.date
    print(
        f"Public LB: {public}"
        f"Date: {date}"
    )
else:
    print("Subs not found")

Public LB: 0.97599Date: 2025-10-05 13:20:14.643000


In [52]:
# === Record Public Score in WandB===
run = wandb.init(
    project=COMP,
    id=WANDB_ID,
    resume="must",
    dir="../../artifacts"
)
run.summary["public"] = public

run.config.update({"ref": ref})

run.finish()

auc_f1,0.97581
auc_f2,0.97503
auc_f3,0.97432
auc_f4,0.97544
auc_f5,0.97503
auc_mean,0.97513
auc_oof,0.97512
auc_std,0.0005
epoch_f1,56
epoch_f2,54
epoch_f3,49


In [53]:
# === Update Manifest ===
m["submission"]["ref"] = ref
m["submission"]["file"] = str(SUB_PATH)
m["submission"]["competition"] = COMP
m["submission"]["public_score"] = public
m["submission"]["submitted_at"] = submitted_at

with open(MANIFEST_PATH, "w") as f:
    json.dump(m, f, indent=4)

# Private Score

In [54]:
subs = api.competition_submissions(
    COMP,
    group=SubmissionGroup.SUBMISSION_GROUP_SUCCESSFUL
)

target = next((s for s in subs if s.ref == ref), None)

private = target.private_score

if target:
    print(f"Private LB: {private}")
else:
    print("Subs not found")

Private LB: 0.97572


In [55]:
# === Record Private Score in WandB===
run = wandb.init(
    project=COMP,
    id=WANDB_ID,
    resume="must",
    dir="../../artifacts"
)
run.summary["private"] = private

run.finish()

auc_f1,0.97581
auc_f2,0.97503
auc_f3,0.97432
auc_f4,0.97544
auc_f5,0.97503
auc_mean,0.97513
auc_oof,0.97512
auc_std,0.0005
epoch_f1,56
epoch_f2,54
epoch_f3,49


In [56]:
# === Update Manifest ===
m["submission"]["private_score"] = private
with open(MANIFEST_PATH, "w") as f:
    json.dump(m, f, indent=4)